## LunarLander-v3

The agent controls a lander that must land safely on a landing pad at the surface of the moon.

### State Space — Continuous, `Box(8,)`

Each state is a single vector of 8 continuous features:

| Index | Feature | Min | Max | Description |
|-------|---------|-----|-----|-------------|
| 0 | X position | -2.5 | 2.5 | Horizontal coordinate of the lander |
| 1 | Y position | -2.5 | 2.5 | Vertical coordinate of the lander |
| 2 | X velocity | -10.0 | 10.0 | Horizontal speed |
| 3 | Y velocity | -10.0 | 10.0 | Vertical speed |
| 4 | Angle | -2π | 2π | Orientation in radians |
| 5 | Angular velocity | -10.0 | 10.0 | Rotational speed |
| 6 | Left leg contact | 0.0 | 1.0 | Whether left leg is touching ground |
| 7 | Right leg contact | 0.0 | 1.0 | Whether right leg is touching ground |

### Action Space — `Discrete(4)`

| Action | Description |
|--------|-------------|
| 0 | Do nothing |
| 1 | Fire left orientation engine |
| 2 | Fire main engine (bottom thruster) |
| 3 | Fire right orientation engine |

### Reward

Potential-based shaping: `reward = shaping - prev_shaping`, where:

```
shaping = -100 * sqrt(x² + y²) - 100 * sqrt(vx² + vy²) - 100 * |angle| + 10 * left_leg + 10 * right_leg
```

- Main engine fuel cost: **-0.30** per frame
- Side engine fuel cost: **-0.03** per frame
- Crash / out-of-bounds: **-100**
- Successful landing (body at rest): **+100**
- Solved threshold: cumulative reward ≥ **200**

### Episode End

| Condition | Type |
|-----------|------|
| Lander crashes (body contacts surface) | Terminated |
| Lander goes out of bounds | Terminated |
| Lander comes to rest (successful landing) | Terminated |
| Episode exceeds 1000 steps | Truncated |

### Starting State

Lander starts at top-center of viewport, upright (angle = 0). A random initial force in range (-1000, 1000) is applied to the center of mass in both x and y directions.

In [6]:
import gymnasium as gym
import csv
import os

gamma = 0.99  # discount factor

# Initialise the environment
env = gym.make("LunarLander-v3", render_mode="human")

# Reset the environment to generate the first observation
observation, info = env.reset(seed=42)
episode = 0
episode_buffer = []

output_path = os.path.join(os.path.dirname(__file__) if '__file__' in dir() else '.', 'output', 'LunarLander-v3.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Episode', 'Experience', 'State', 'Action', 'Reward', 'Terminated', 'Truncated', 'Return'])

    for experience in range(1000):
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        episode_buffer.append([episode, experience, observation.tolist(), action, reward, terminated, truncated])

        if terminated or truncated:
            # Compute returns backwards
            G = 0
            for i in reversed(range(len(episode_buffer))):
                G = episode_buffer[i][4] + gamma * G
                episode_buffer[i].append(round(G, 4))
            for row in episode_buffer:
                writer.writerow(row)
            episode_buffer = []
            episode += 1
            observation, info = env.reset()

    # Write any remaining experiences from an incomplete episode
    if episode_buffer:
        G = 0
        for i in reversed(range(len(episode_buffer))):
            G = episode_buffer[i][4] + gamma * G
            episode_buffer[i].append(round(G, 4))
        for row in episode_buffer:
            writer.writerow(row)

env.close()
print(f"CSV written to {output_path}")

CSV written to ./output/LunarLander-v3.csv


## CartPole-v1

The agent balances a pole hinged on a cart by pushing the cart left or right. The goal is to keep the pole upright for as long as possible.

### State Space — Continuous, `Box(4,)`

Each state is a single vector of 4 continuous features:

| Index | Feature | Min | Max | Termination Threshold |
|-------|---------|-----|-----|-----------------------|
| 0 | Cart position | -4.8 | 4.8 | ±2.4 |
| 1 | Cart velocity | -Inf | Inf | — |
| 2 | Pole angle (rad) | -0.418 | 0.418 | ±0.2095 (~12°) |
| 3 | Pole angular velocity | -Inf | Inf | — |

Note: the observation space bounds are wider than the termination thresholds.

### Action Space — `Discrete(2)`

| Action | Description |
|--------|-------------|
| 0 | Push cart to the left |
| 1 | Push cart to the right |

### Reward

- **+1** for every step taken (including the termination step)
- Maximum possible reward: **500** (surviving all steps)
- Solved threshold: cumulative reward ≥ **500**

### Episode End

| Condition | Type |
|-----------|------|
| Pole angle exceeds ±12° | Terminated |
| Cart position exceeds ±2.4 | Terminated |
| Episode exceeds 500 steps | Truncated |

### Starting State

All 4 observation values are sampled uniformly from the range **(-0.05, 0.05)**.

In [7]:
import gymnasium as gym
import csv
import os

gamma = 0.99  # discount factor

# Initialise the environment
env = gym.make("CartPole-v1", render_mode="human")

# Reset the environment to generate the first observation
observation, info = env.reset(seed=42)
episode = 0
episode_buffer = []

output_path = os.path.join(os.path.dirname(__file__) if '__file__' in dir() else '.', 'output', 'CartPole-v1.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Episode', 'Experience', 'State', 'Action', 'Reward', 'Terminated', 'Truncated', 'Return'])

    for experience in range(1000):
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        episode_buffer.append([episode, experience, observation.tolist(), action, reward, terminated, truncated])

        if terminated or truncated:
            # Compute returns backwards
            G = 0
            for i in reversed(range(len(episode_buffer))):
                G = episode_buffer[i][4] + gamma * G
                episode_buffer[i].append(round(G, 4))
            for row in episode_buffer:
                writer.writerow(row)
            episode_buffer = []
            episode += 1
            observation, info = env.reset()

    # Write any remaining experiences from an incomplete episode
    if episode_buffer:
        G = 0
        for i in reversed(range(len(episode_buffer))):
            G = episode_buffer[i][4] + gamma * G
            episode_buffer[i].append(round(G, 4))
        for row in episode_buffer:
            writer.writerow(row)

env.close()
print(f"CSV written to {output_path}")

CSV written to ./output/CartPole-v1.csv


## MountainCar-v0

A car is stuck in a valley between two hills. The engine is too weak to climb directly — the agent must learn to rock back and forth to build momentum and reach the goal on the right hilltop.

### State Space — Continuous, `Box(2,)`

Each state is a single vector of 2 continuous features:

| Index | Feature | Min | Max | Description |
|-------|---------|-----|-----|-------------|
| 0 | Car position | -1.2 | 0.6 | Position along the x-axis |
| 1 | Car velocity | -0.07 | 0.07 | Speed of the car |

Both values are clipped to their bounds each step. If the car hits the left boundary (-1.2), velocity resets to 0 (inelastic collision).

### Action Space — `Discrete(3)`

| Action | Description |
|--------|-------------|
| 0 | Accelerate to the left |
| 1 | Don't accelerate |
| 2 | Accelerate to the right |

### Reward

- **-1** for every timestep
- No bonus for reaching the goal — the only way to maximise return is to reach the goal as quickly as possible

### Episode End

| Condition | Type |
|-----------|------|
| Car position ≥ 0.5 (goal reached) | Terminated |
| Episode exceeds 200 steps | Truncated |

### Starting State

- Position sampled uniformly from **[-0.6, -0.4]**
- Velocity always initialised to **0**

In [8]:
import gymnasium as gym
import csv
import os

gamma = 0.99  # discount factor

# Initialise the environment
env = gym.make("MountainCar-v0", render_mode="human")

# Reset the environment to generate the first observation
observation, info = env.reset(seed=42)
episode = 0
episode_buffer = []

output_path = os.path.join(os.path.dirname(__file__) if '__file__' in dir() else '.', 'output', 'MountainCar-v0.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Episode', 'Experience', 'State', 'Action', 'Reward', 'Terminated', 'Truncated', 'Return'])

    for experience in range(1000):
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        episode_buffer.append([episode, experience, observation.tolist(), action, reward, terminated, truncated])

        if terminated or truncated:
            # Compute returns backwards
            G = 0
            for i in reversed(range(len(episode_buffer))):
                G = episode_buffer[i][4] + gamma * G
                episode_buffer[i].append(round(G, 4))
            for row in episode_buffer:
                writer.writerow(row)
            episode_buffer = []
            episode += 1
            observation, info = env.reset()

    # Write any remaining experiences from an incomplete episode
    if episode_buffer:
        G = 0
        for i in reversed(range(len(episode_buffer))):
            G = episode_buffer[i][4] + gamma * G
            episode_buffer[i].append(round(G, 4))
        for row in episode_buffer:
            writer.writerow(row)

env.close()
print(f"CSV written to {output_path}")

CSV written to ./output/MountainCar-v0.csv


## Acrobot-v1

A two-link pendulum (acrobot) hangs downward from a fixed point. The agent applies torque at the joint between the two links. The goal is to swing the free end above a target height.

### State Space — Continuous, `Box(6,)`

Each state is a single vector of 6 continuous features:

| Index | Feature | Min | Max | Description |
|-------|---------|-----|-----|-------------|
| 0 | cos(θ₁) | -1.0 | 1.0 | Cosine of first joint angle |
| 1 | sin(θ₁) | -1.0 | 1.0 | Sine of first joint angle |
| 2 | cos(θ₂) | -1.0 | 1.0 | Cosine of second joint angle (relative to link 1) |
| 3 | sin(θ₂) | -1.0 | 1.0 | Sine of second joint angle (relative to link 1) |
| 4 | Angular velocity of θ₁ | -12.57 (-4π) | 12.57 (4π) | Rotational speed of first link |
| 5 | Angular velocity of θ₂ | -28.27 (-9π) | 28.27 (9π) | Rotational speed of second link |

θ₁ = 0 means the first link points straight down. θ₂ is relative to the first link (0 means same direction).

### Action Space — `Discrete(3)`

| Action | Description |
|--------|-------------|
| 0 | Apply -1 torque |
| 1 | Apply 0 torque (no torque) |
| 2 | Apply +1 torque |

### Reward

- **-1** for every step that does not reach the goal
- **0** on the step the goal is reached (episode terminates)
- Solved threshold: cumulative reward ≥ **-100**

### Episode End

| Condition | Type |
|-----------|------|
| Free end reaches target height: `-cos(θ₁) - cos(θ₂ + θ₁) > 1.0` | Terminated |
| Episode exceeds 500 steps | Truncated |

### Starting State

All 4 internal state values (θ₁, θ₂, angular velocity of θ₁, angular velocity of θ₂) are sampled uniformly from **[-0.1, 0.1]**.

In [9]:
import gymnasium as gym
import csv
import os

gamma = 0.99  # discount factor

# Initialise the environment
env = gym.make("Acrobot-v1", render_mode="human")

# Reset the environment to generate the first observation
observation, info = env.reset(seed=42)
episode = 0
episode_buffer = []

output_path = os.path.join(os.path.dirname(__file__) if '__file__' in dir() else '.', 'output', 'Acrobot-v1.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Episode', 'Experience', 'State', 'Action', 'Reward', 'Terminated', 'Truncated', 'Return'])

    for experience in range(1000):
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        episode_buffer.append([episode, experience, observation.tolist(), action, reward, terminated, truncated])

        if terminated or truncated:
            # Compute returns backwards
            G = 0
            for i in reversed(range(len(episode_buffer))):
                G = episode_buffer[i][4] + gamma * G
                episode_buffer[i].append(round(G, 4))
            for row in episode_buffer:
                writer.writerow(row)
            episode_buffer = []
            episode += 1
            observation, info = env.reset()

    # Write any remaining experiences from an incomplete episode
    if episode_buffer:
        G = 0
        for i in reversed(range(len(episode_buffer))):
            G = episode_buffer[i][4] + gamma * G
            episode_buffer[i].append(round(G, 4))
        for row in episode_buffer:
            writer.writerow(row)

env.close()
print(f"CSV written to {output_path}")

CSV written to ./output/Acrobot-v1.csv


## Taxi-v3

A taxi navigates a 5×5 grid to pick up a passenger at one of 4 locations and drop them off at another. The grid has walls that block movement.

```
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
```

`|` between cells = wall, `:` = open passage. **R** (Red), **G** (Green), **Y** (Yellow), **B** (Blue) are the pickup/dropoff locations.

### State Space — Discrete, `Discrete(500)`

The state is a single integer that encodes 4 components:

| Component | # of values | Range | Meaning |
|-----------|-------------|-------|---------|
| Taxi row | 5 | 0–4 | Row on the 5×5 grid |
| Taxi col | 5 | 0–4 | Column on the 5×5 grid |
| Passenger location | 5 | 0–4 | 0=Red, 1=Green, 2=Yellow, 3=Blue, 4=In taxi |
| Destination | 4 | 0–3 | 0=Red, 1=Green, 2=Yellow, 3=Blue |

Total: 5 × 5 × 5 × 4 = **500 states** (400 reachable during normal play).

#### Encoding — packing 4 values into 1 integer

The idea is the same as flattening a multi-dimensional array into a 1D index. Each component gets its own "digit" in a mixed-radix number system (bases 5, 5, 5, 4 instead of 10, 10, 10, 10).

```
state = ((taxi_row × 5 + taxi_col) × 5 + passenger_loc) × 4 + destination
```

Building it step by step:

1. **Flatten the grid position** — `taxi_row × 5 + taxi_col` maps (row, col) to a number 0–24. Multiplying by 5 (the number of columns) "shifts" the row left to make room for the column.
2. **Add passenger location** — `× 5 + passenger_loc` multiplies the result so far by 5 (the number of passenger values) to make room, then adds the passenger value. Now we have a unique number for every (position, passenger) combo.
3. **Add destination** — `× 4 + destination` multiplies by 4 (the number of destinations) to make room, then adds the destination. Now every possible configuration maps to a unique integer 0–499.

**Example — encode** taxi at row=1, col=3, passenger=Yellow (2), destination=Blue (3):

```
step 1:  1 × 5 + 3  =  8        ← grid position (row=1, col=3)
step 2:  8 × 5 + 2  = 42        ← with passenger location (Yellow=2)
step 3: 42 × 4 + 3  = 171       ← with destination (Blue=3)
```

State = **171**

#### Decoding — unpacking the integer back to 4 values

Reverse the encoding using modulo (`%`) and integer division (`//`). Peel off each component from right to left (destination first, then passenger, then col, then row):

```
destination    = state % 4
remainder      = state // 4
passenger_loc  = remainder % 5
remainder      = remainder // 5
taxi_col       = remainder % 5
taxi_row       = remainder // 5
```

**Example — decode** state = 171:

```
step 1: 171 % 4  = 3   → destination = 3 (Blue)
        171 // 4 = 42
step 2:  42 % 5  = 2   → passenger   = 2 (Yellow)
         42 // 5 = 8
step 3:   8 % 5  = 3   → taxi_col    = 3
          8 // 5 = 1   → taxi_row    = 1
```

Result: taxi at (1, 3), passenger at Yellow, destination Blue — matches the original.

### Action Space — `Discrete(6)`

| Action | Description |
|--------|-------------|
| 0 | Move south (down) |
| 1 | Move north (up) |
| 2 | Move east (right) |
| 3 | Move west (left) |
| 4 | Pickup passenger |
| 5 | Drop off passenger |

Moving into a wall keeps the taxi in place but still incurs the step penalty.

### Reward

| Event | Reward |
|-------|--------|
| Every timestep | **-1** |
| Successful dropoff at correct destination | **+20** |
| Illegal pickup or dropoff | **-10** |

### Episode End

| Condition | Type |
|-----------|------|
| Passenger dropped off at correct destination | Terminated |
| Episode exceeds 200 steps | Truncated |

### Starting State

Sampled uniformly from 300 valid configurations: 25 taxi positions × 4 passenger locations × 3 non-matching destinations (passenger not already at destination and not in taxi).

In [11]:
import gymnasium as gym
import csv
import os

gamma = 0.99  # discount factor

# Initialise the environment
env = gym.make("Taxi-v3", render_mode="human")

# Reset the environment to generate the first observation
observation, info = env.reset(seed=42)
episode = 0
episode_buffer = []

output_path = os.path.join(os.path.dirname(__file__) if '__file__' in dir() else '.', 'output', 'Taxi-v3.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Episode', 'Experience', 'State', 'Action', 'Reward', 'Terminated', 'Truncated', 'Return'])

    for experience in range(100):
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        episode_buffer.append([episode, experience, observation, action, reward, terminated, truncated])

        if terminated or truncated:
            # Compute returns backwards
            G = 0
            for i in reversed(range(len(episode_buffer))):
                G = episode_buffer[i][4] + gamma * G
                episode_buffer[i].append(round(G, 4))
            for row in episode_buffer:
                writer.writerow(row)
            episode_buffer = []
            episode += 1
            observation, info = env.reset()

    # Write any remaining experiences from an incomplete episode
    if episode_buffer:
        G = 0
        for i in reversed(range(len(episode_buffer))):
            G = episode_buffer[i][4] + gamma * G
            episode_buffer[i].append(round(G, 4))
        for row in episode_buffer:
            writer.writerow(row)

env.close()
print(f"CSV written to {output_path}")

CSV written to ./output/Taxi-v3.csv
